# PoC LKP P6 — CowCare
**Tujuan:** membuktikan pipeline klasifikasi gejala PMK (MobileNetV2) bisa berjalan sebelum coding UI.

- Dataset: `FMD.v1i.yolov8.zip` (proyek Roboflow kelompok, CC BY 4.0)
- 4 kelas proxy: `mulut_sakit`, `mulut_sehat`, `kuku_sakit`, `kuku_sehat`
- Split: mengikuti Roboflow, lalu duplikat eksak lintas-split dibuang
- Output: accuracy, macro-F1, recall kelas sakit, false-negative rate, confusion matrix
- Catatan: ini PoC 4 kelas, **bukan** klaim 13 kelas.


In [ ]:
# ============================================================
# PoC LKP P6 - CowCare | MobileNetV2 4 kelas proxy
# Dataset: FMD.v1i.yolov8.zip (proyek Roboflow kelompok, CC BY 4.0)
# Jalankan di Google Colab: Runtime > Change runtime type > CPU/GPU
# ============================================================
!pip install -q torch torchvision scikit-learn

import os, json, hashlib, random, time, zipfile
from collections import Counter
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(max(1,(os.cpu_count() or 4)-1))

# --- 1. Upload dataset (pilih FMD.v1i.yolov8.zip) ---
from google.colab import files
up = files.upload()
ZIP = list(up.keys())[0]
WORK = '/content/data'
with zipfile.ZipFile(ZIP) as z: z.extractall(WORK)

NAMES = ['kuku_sakit','kuku_sehat','mulut_sakit','mulut_sehat']

# --- 2. Index + dedup eksak lintas split ---
def index_split(split):
    img_dir=os.path.join(WORK,split,'images'); lbl_dir=os.path.join(WORK,split,'labels')
    out=[]
    for fn in sorted(os.listdir(img_dir)):
        if not fn.lower().endswith(('.jpg','.jpeg','.png')): continue
        lp=os.path.join(lbl_dir,os.path.splitext(fn)[0]+'.txt')
        if not os.path.exists(lp): continue
        best=(0.0,-1)
        for line in open(lp,encoding='utf-8',errors='ignore'):
            p=line.split()
            if len(p)==5:
                a=float(p[3])*float(p[4])
                if a>best[0]: best=(a,int(float(p[0])))
        if best[1]<0: continue
        fp=os.path.join(img_dir,fn)
        md5=hashlib.md5(open(fp,'rb').read()).hexdigest()
        out.append({'path':fp,'label':best[1],'area':best[0],'md5':md5,'split':split})
    return out

all_s=[]
for sp in ['train','valid','test']:
    s=index_split(sp); print(sp,len(s),Counter(x['label'] for x in s)); all_s+=s

seen=set(); keep=[]; leak=0
for sp in ['train','valid','test']:
    for r in [x for x in all_s if x['split']==sp]:
        if r['md5'] in seen: leak+=1; continue
        seen.add(r['md5']); keep.append(r)
print('duplikat lintas-split dibuang:',leak,'bersih:',len(keep))

# --- 3. Dataset & transform ---
class FMDDataset(Dataset):
    def __init__(self,rows,tf): self.rows=rows; self.tf=tf
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        return self.tf(Image.open(r['path']).convert('RGB')), r['label'], i

train_tf=transforms.Compose([
    transforms.RandomResizedCrop(224,scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
eval_tf=transforms.Compose([
    transforms.Resize(256),transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

tr=[r for r in keep if r['split']=='train']
va=[r for r in keep if r['split']=='valid']
te=[r for r in keep if r['split']=='test']
bs=32
dl_tr=DataLoader(FMDDataset(tr,train_tf),batch_size=bs,shuffle=True)
dl_va=DataLoader(FMDDataset(va,eval_tf),batch_size=bs,shuffle=False)
dl_te=DataLoader(FMDDataset(te,eval_tf),batch_size=bs,shuffle=False)

# --- 4. Model ---
model=mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier[1]=nn.Linear(model.classifier[1].in_features,len(NAMES))
for p in model.features.parameters(): p.requires_grad=False
for blk in model.features[-4:]:
    for p in blk.parameters(): p.requires_grad=True

crit=nn.CrossEntropyLoss(label_smoothing=0.05)
head=list(model.classifier.parameters())
back=[p for p in model.parameters() if p.requires_grad and not any(p is q for q in head)]
opt=torch.optim.AdamW([{'params':head,'lr':1e-3},{'params':back,'lr':1e-4}],weight_decay=1e-4)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=7)

def evaluate(dl):
    model.eval(); ys=[]; ps=[]; idxs=[]
    with torch.no_grad():
        for x,y,i in dl:
            logit=model(x)
            ps+=logit.argmax(1).tolist(); ys+=y.tolist(); idxs+=i.tolist()
    return np.array(ys),np.array(ps),idxs

# --- 5. Training 7 epoch ---
for ep in range(7):
    model.train(); tot=corr=0; ls=0.0
    for x,y,_ in dl_tr:
        opt.zero_grad(); logit=model(x); loss=crit(logit,y); loss.backward(); opt.step()
        ls+=loss.item()*len(y); corr+=(logit.argmax(1)==y).sum().item(); tot+=len(y)
    sched.step()
    yv,pv,_=evaluate(dl_va)
    print(f'epoch {ep+1}/7 loss={ls/tot:.3f} train_acc={corr/tot:.3f} val_acc={(yv==pv).mean():.3f}')

# --- 6. Metrik test ---
y,p,idxs=evaluate(dl_te)
acc=accuracy_score(y,p)
prec,rec,f1,sup=precision_recall_fscore_support(y,p,labels=range(4),zero_division=0)
sick=[i for i,n in enumerate(NAMES) if n.endswith('sakit')]
print(f'TEST acc={acc:.4f} macroF1={f1.mean():.4f} sick_recall={rec[sick].mean():.4f} FN={1-rec[sick].mean():.4f}')
print(classification_report(y,p,target_names=NAMES,digits=3,zero_division=0))
print('confusion:\n',confusion_matrix(y,p,labels=range(4)))
areas=np.array([te[i]['area'] for i in idxs])
for name,m in [('close>=8%',areas>=0.08),('wide<8%',areas<0.08)]:
    if m.sum(): print(name,'n=',int(m.sum()),'acc=',round(float((y[m]==p[m]).mean()),4))
